# Reproducing Makin et al. (2018) from dandiset 000129: decoders at 64 ms

**Paper.** Makin JG, O'Doherty JE, Cardoso MMB, Sabes PN (2018). Superior arm-movement decoding
from cortex with a new, unsupervised-learning algorithm. *J. Neural Eng.* 15, 026010.
[doi:10.1088/1741-2552/aa9e95](https://doi.org/10.1088/1741-2552/aa9e95)

**Dataset.** [DANDI:000129](https://dandiarchive.org/dandiset/000129/0.241017.1444) version
0.241017.1444 (doi:10.48324/dandi.000129/0.241017.1444), "MC_RTT", streamed from the archive. The
paper cites no version of the dandiset; 0.241017.1444 is the only published version (the draft
holds the same two assets, 50,965,512 bytes, and differs only in its title), so it is pinned.

**Claims.** `claims/000129/10.1088-1741-2552-aa9e95.yaml`, written from the version of record
(main text, Methods and legends; the paper has no supplement) and the paper's code
(github.com/jgmakin/rbmish at commit 700062f5).

**What the dataset holds.** One session of monkey I (Indy), M1 array only, from the Neural Latents
Benchmark: a training file with 649.1 s of sorted-unit spikes and kinematics, and a test file with
190.3 s of spikes whose kinematics are withheld. The paper used 49 sessions from three monkeys.
Claims judged on a mean over sessions are tested here on this one session (a deviation); claims
that need the bootstrap across the 49 sessions, or the per-session spread, cannot be tested.

This notebook covers the decoders at 64 ms bins with 320 s of training (figures 2, 4 and 8).
The notebook `binwidth_training_neurons.ipynb` covers bin widths, amounts of training data and
fractions of neurons (figures 5 to 7).

| Claim | Here |
|---|---|
| units-monkey-i | tested |
| fig2-velocity-boost, fig2-kflatent-dynamic-little-effect, fig2-refh-static-vel-acc, fig2-position-dynamic-gain, fig2-sagittal-better, fig2-linear-worst, fig2-gross-failures | tested on one session |
| fig4-vel-acc-every-session (per-session part), fig4-best-variable, kflatent-modest | tested on one session |
| fig8-velocity-variants, fig8-gaussian-still-beats-ukf | tested on one session |
| fig5-advantage-with-binwidth, fig5-derivative-order, fig7-affine-position, fig7-kinematic-model-saturates, advantage-grows-with-derivative | tested in `binwidth_training_neurons.ipynb` |
| fig2-refh-kf-best, intro-gain-over-nearest, fig4-position, fig8-position-variants, static-refh-position-worse | blocked: bootstrap across the paper's 49 sessions, which the dandiset does not hold; the single-session values are printed below |
| fig4-advantage-grows, fig5-vel-acc-all-bins, fig5-position-bins, fig6-training-time, fig7-vel-acc-neurons, fig7-position-neurons, fig9-taps | blocked: bootstrap or regression across the paper's 49 sessions (single-session values of the fig5 to fig7 claims in `binwidth_training_neurons.ipynb`; the tapped decoders of fig9-taps are not implemented) |
| fig4-task-similar | blocked: monkey C's data and monkey I's other sessions |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")
import time
import numpy as np
import pandas as pd
import requests
import makin as M

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
d = M.load_session()
print(f"dandiset {M.DANDISET} version {M.VERSION}")
print(d["session_description"])
print("session_start_time", d["session_start_time"])

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
ids = d["unit_ids"]
n_spk = np.array([len(s) for s in d["spikes"]])
dur = len(d["finger_pos"]) / d["rate"]
print(f"subjects 1 (Indy), sessions 1, units {len(ids)} ({d['heldout'].sum()} marked heldout by the benchmark)")
print(f"kinematics: {len(d['finger_pos'])} samples at {d['rate']:.0f} Hz from t0 = {d['t0']} s, duration {dur:.1f} s")
print("finger_pos unit attribute:", d["unit"], " conversion:", d["conversion"])
for k in ["finger_pos", "cursor_pos", "target_pos"]:
    a = d[k]
    print(f"{k:11s} x {np.nanmin(a[:,0]):7.1f} to {np.nanmax(a[:,0]):6.1f}, y {np.nanmin(a[:,1]):6.1f} to {np.nanmax(a[:,1]):6.1f} (stored values); NaN samples {np.isnan(a[:,0]).sum()}")
gaps = M.segments(np.isnan(d["finger_pos"][:, 0]))
print("NaN gaps (s):", [(round(s / d['rate'], 3), round(e / d['rate'], 3)) for s, e in gaps])
st_all = np.concatenate(d["spikes"])
print(f"spikes {len(st_all)}, times {st_all.min():.3f} to {st_all.max():.3f} s, sorted within units: {all(np.all(np.diff(s) >= 0) for s in d['spikes'])}")
in_gap = sum(((st_all >= s / d['rate']) & (st_all < e / d['rate'])).sum() for s, e in gaps)
print("spikes inside the NaN gaps:", in_gap)
print("electrode locations:", pd.Series(d["electrode_locations"]).value_counts().to_dict())
print("successive 1 kHz finger samples that are equal:", int((np.diff(d["finger_pos"][:, 0]) == 0).sum()))
print("target x positions (stored):", np.unique(d["target_pos"][~np.isnan(d["target_pos"][:, 0]), 0]))
print("unit ids end in:", sorted(set(int(i) % 100 for i in ids)), "(channel*100 + unit; 0 would be the unsorted hash unit)")
print("channels with units:", len(set(int(i) // 100 for i in ids)))
rate = n_spk / dur
print(f"rates (Hz) min {rate.min():.2f}, median {np.median(rate):.2f}, max {rate.max():.2f}")
print("trials table splits:", pd.Series(d["trials_split"]).value_counts().to_dict())

The kinematic series are stored in mm with unit "meters" and conversion 0.001, which is
consistent (value times conversion gives meters); the values span about 135 by 125 mm, and the targets sit
at eight x positions 15 mm apart, as the 8 by 8 grid of the description and of the paper's later
monkey I sessions has. All electrodes are labelled M1. The finger
position description says 250 Hz upsampled to 1 kHz; successive 1 kHz samples are never equal,
so the upsampling interpolated, and taking every fourth sample recovers a 250 Hz series. Three
gaps of 0.1 s are NaN in every behavioural series. The unit ids end in 1 to 4: no unsorted hash
unit is in the file, though the paper kept one per channel (Methods, Data acquisition).

In [ ]:
# Is this one of the paper's sessions? The paper's session list (code, NeuralAnalysis/Indy_HC_Sessions.csv)
CSV = ("https://raw.githubusercontent.com/jgmakin/rbmish/700062f5118b0910411cf9a8f73498e22203731d/"
       "NeuralAnalysis/Indy_HC_Sessions.csv")
sess = pd.read_csv(CSV, header=None, names=["date", "block", "n_reaches", "arrays", "grid", "use"])
print(f"{len(sess)} rows; dates {sess.date.min()} to {sess.date.max()}; rows with at least 300 reaches: {(sess.n_reaches >= 300).sum()}")
print("2017-02-02 in the list:", 20170202 in set(sess.date))
print(sess.tail(4).to_string(index=False))

The file dates the session 2017-02-02. The paper's session list ends on 2017-01-31 and does not
hold that date, so the session in the dandiset is probably not one of the paper's 37 monkey I
sessions (it could also be a misdated one; nothing in the file links it to the list). It
matches the paper's description of monkey I's later sessions: M1 array only, 8 by 8 grid.

In [ ]:
# Test file: spikes only (kinematics withheld), and whether it repeats any of the training file
ft = M.open_asset(M.TEST_ASSET)
tid = ft["units/id"][:]
tst = np.split(ft["units/spike_times"][:], ft["units/spike_times_index"][:-1])
print("test file units:", len(tid), " behaviour in test file:", "processing" in ft and len(ft["processing"]) > 0)
print(f"test file spikes {sum(len(s) for s in tst)}, span {min(s.min() for s in tst if len(s)):.2f} to {max(s.max() for s in tst if len(s)):.2f} s")
# duplicated recording check: runs of 4 inter-spike intervals (to 1 us) shared between the files, per unit
idmap = {int(i): k for k, i in enumerate(ids)}
shared = total = 0
for i, s in zip(tid, tst):
    a = d["spikes"][idmap[int(i)]]
    keys = lambda x: set(map(tuple, np.round(np.lib.stride_tricks.sliding_window_view(np.diff(x), 4) * 1e6).astype(np.int64)))
    kt = keys(s); total += len(kt); shared += len(kt & keys(a))
print(f"4-ISI patterns in the test file {total}, also in the training file {shared}")
ft.close()

The test file holds 98 of the 130 units (the benchmark's held-in units) and no kinematics, so it
cannot be scored; only the training file is used. No run of four inter-spike intervals is shared
between the files, so the test file does not repeat the training file.

In [ ]:
# Clock check: decode velocity from spike counts with the kinematics shifted in time.
R, X, edges = M.bin_data(d, 64)
keep = M.unit_filter(R, 64, 5000)
ok = ~np.isnan(X).any(1)
rows = []
for lag in range(-8, 9):  # kinematics at bin t+lag decoded from spikes at bin t
    Xs = np.roll(X, -lag, axis=0)
    okl = ok & np.roll(ok, -lag) & (np.arange(len(X)) >= 8) & (np.arange(len(X)) < len(X) - 8)
    tr = okl & (np.arange(len(X)) < 5000); te = okl & (np.arange(len(X)) >= 5000)
    B = M.ridge(R[tr][:, keep], Xs[tr])
    r2 = M.rsq(M.apply(B, R[te][:, keep]), Xs[te])
    rows.append(dict(lag_ms=lag * 64, R2_vx=r2[2], R2_vy=r2[3], R2_x=r2[0]))
lagdf = pd.DataFrame(rows)
print(lagdf.round(3).to_string(index=False))
best = lagdf.loc[lagdf[["R2_vx", "R2_vy"]].mean(axis=1).idxmax(), "lag_ms"]
print("best lag for velocity (ms):", best)

Velocity is decoded best when the kinematics lag the spikes by about one to two bins, and the
fit falls off on both sides; the spikes lead the movement, as motor cortex does, so the spike
times and the kinematics share one clock. The paper's decoders use no offset (Methods, KFobs),
and neither does this attempt.

In [ ]:
print(f"bins of 64 ms: {len(X)}; training bins (first 320 s) 5000; test bins {len(X) - 5000}")
print(f"bins with NaN kinematics: {(~ok).sum()} (training {(~ok[:5000]).sum()}, test {(~ok[5000:]).sum()})")
rate64 = R.mean(0) / 0.064
dup = (rate64 > 0.5).sum() - keep.sum()
print(f"units: {R.shape[1]} in file, {(rate64 > 0.5).sum()} above 0.5 Hz, {dup} dropped as duplicates, {keep.sum()} used")
Xk = X[ok]
for i, v in enumerate(M.VARS):
    print(f"{v:2s}: {Xk[:, i].min():9.1f} to {Xk[:, i].max():8.1f}  (mm, mm/s, mm/s^2); test SD {np.nanstd(X[5000:, i]):.1f}")

## Decoders at 64 ms

All eight decoders of the paper are fitted on the first 320 s and tested on the remaining 329 s
(the Methods, "Training and testing data"). The decoders and where each detail comes from are in
the docstrings of `makin.py`. The kinematics are the Methods reading (fingertip, 10 Hz
Butterworth, last sample of each bin); the code's reading (cursor, no filter, bin means) is run
as a variant below. Scores: R2 per variable on the test bins, and SNR = -10 log10(1 - R2) in dB.

In [ ]:
def fit_config(seed=0, **kw):
    sp = M.prepare(d, **kw)
    t = time.time()
    info = {}
    pred = M.run_all(sp, seed, info=info)
    r2 = pd.DataFrame({k: sp.score(v) for k, v in pred.items()}, index=M.VARS).T.loc[M.DECODERS]
    return sp, pred, r2, time.time() - t, info

sp, pred, r2, secs, info = fit_config(seed=0)
print(f"fitted in {secs:.0f} s; units {sp.Rtr.shape[1]}, training bins {len(sp.Rtr)} ({sp.oktr.sum()} scored), test bins {len(sp.Rte)} ({sp.okte.sum()} scored)")
snr = M.snr(r2)
print("R2:"); print(r2.round(3).to_string())
print("\nSNR (dB):"); print(snr.round(2).to_string())
kfi = info["kflatent"]
print(f"\nKFlatent EM: {kfi['k']} latent states, {kfi['epochs']} E steps, log-likelihood {kfi['ll_first']:.0f} to {kfi['ll_last']:.0f}; "
      f"gain of the last epoch {kfi['last_rel_gain']:.2e} of the total (criterion 1e-4); met the criterion: {bool(kfi['converged'])}")
print("static rEFH map, R2 on the training bins it was fitted on:", dict(zip(M.VARS, info["refh_train_r2"].round(3))))
print("static rEFH map, R2 on the test bins:                     ", dict(zip(M.VARS, r2.loc["refh"].values.round(3))))


In [ ]:
def summary(snr):
    # SNR averaged over x and y per kinematic variable (Results: summaries average x and y)
    return pd.DataFrame({"pos": snr[["x", "y"]].mean(axis=1), "vel": snr[["vx", "vy"]].mean(axis=1),
                         "acc": snr[["ax", "ay"]].mean(axis=1)})
S = summary(snr)
print(S.round(2).to_string())

KFlatent's EM ran all 100 epochs: the last epoch raised the log-likelihood by 4.25e-04 of the
total gain, above the Methods' 1/100 of a percent (1e-4), so the 100-epoch limit stopped it, as the
Methods allow. The static rEFH map fits its training bins better than the test bins (R2 0.39 and
0.51 against 0.19 and 0.37 for x and y position, 0.67 and 0.71 against 0.59 and 0.59 for
velocity); the second KF of the rEFH (+KF) is fitted on these training outputs, as the paper's
dynamic map is.

### Variants run for every claim of this notebook

The rEFH and the EM of KFlatent are stochastic and the paper reports one training run per
session; seed 0 is the main run (a standard choice, as the claims file says). Two more rEFH
seeds, and two more seeds of the factor-analysis initialization of KFlatent's EM, show how much a
claim depends on the run. The code's kinematics (cursor position, no
filter, bin means) are the variant the claims file names.

In [ ]:
variants = {}
for s in (1, 2):
    sp_s = M.prepare(d)
    t = time.time()
    r = M.dec_refh(sp_s, seed=10 * s + 1)
    v = r2.copy()
    v.loc["refh"], v.loc["refh_kf"] = sp_s.score(r["refh"]), sp_s.score(r["refh_kf"])
    variants[f"rEFH seed {s}"] = M.snr(v)
    print(f"rEFH seed {s}: {time.time() - t:.0f} s")
for s in (1, 2):
    sp_s = M.prepare(d)
    t = time.time()
    st, dyn, kfi_s = M.dec_kflatent(sp_s, seed=10 * s, return_info=True)
    v = r2.copy()
    v.loc["kflatent"], v.loc["kflatent_kf"] = sp_s.score(st), sp_s.score(dyn)
    variants[f"KFlatent seed {s}"] = M.snr(v)
    print(f"KFlatent seed {s}: {kfi_s['epochs']} E steps, last gain {kfi_s['last_rel_gain']:.2e}, {time.time() - t:.0f} s")
sp_c, pred_c, r2_c, secs, info_c = fit_config(seed=0, reading="code")
variants["code kinematics"] = M.snr(r2_c)
print(f"code kinematics: {secs:.0f} s; KFlatent EM {info_c['kflatent']['epochs']} E steps, last gain {info_c['kflatent']['last_rel_gain']:.2e}")
for k, v in variants.items():
    print(f"\n{k}, SNR averaged over x and y:"); print(summary(v).round(2).to_string())

## units-monkey-i

> After units firing below 0.5 Hz were discarded, monkey I's sessions had between 171 and 413 units, the range reflecting that only 7 of 37 sessions included the S1 array.

In [ ]:
rate_session = n_spk / dur
n_cut = int((rate_session > 0.5).sum())
print(f"units in the file {len(ids)}; above 0.5 Hz over the session {n_cut}; at or below 0.5 Hz {len(ids) - n_cut}")
print(f"criterion range 153.9 to 454.3; inside: {153.9 <= n_cut <= 454.3}")
print("hash units in the file: 0 (no unit id ends in 00); channels with sorted units:", len(set(int(i) // 100 for i in ids)))
for thr in (0.25, 0.5, 1.0):
    print(f"  threshold {thr} Hz: {(rate_session > thr).sum()} units")

**Result: not reproduced.** 111 of the 130 units in the file fire above 0.5 Hz over the session, below the 153.9 to 454.3 range. The file holds sorted units only (no unit id ends in 00), on 82 channels. The paper's counts include an unsorted hash unit for each channel (Methods, Data acquisition), up to 96 more here, so the shortfall is consistent with the missing hash units.

## fig2-velocity-boost

> Relative to the static linear map, the rEFH (+KF) raises velocity SNR by more than twice as much as KFobs and nearly twice as much as the UKF.

In [ ]:
def velocity_boost(snr):
    S = summary(snr)
    g = S["vel"] - S.loc["linear", "vel"]
    rk, ru = g["refh_kf"] / g["kfobs"], g["refh_kf"] / g["ukf"]
    return dict(gain_refh_kf=g["refh_kf"], gain_kfobs=g["kfobs"], gain_ukf=g["ukf"], ratio_kfobs=rk, ratio_ukf=ru,
                kfobs_part=rk > 2, ukf_part=1.5 <= ru <= 2.2)
rows = {"main": velocity_boost(snr), **{k: velocity_boost(v) for k, v in variants.items()}}
print(pd.DataFrame(rows).T.round(2).to_string())

**Result: not reproduced, resting on the Methods reading of the kinematics.** The rEFH (+KF) raises velocity SNR over linear regression by 3.67 dB, KFobs by 1.91 dB and the UKF by 2.42 dB. That is 1.92 times KFobs's gain, where the criterion needs more than 2, and 1.51 times the UKF's, inside 1.5 to 2.2. With the code's kinematics the ratios are 2.05 and 1.70 and the claim holds. The other two rEFH seeds give KFobs ratios of 1.93 and 1.95.

## fig2-kflatent-dynamic-little-effect

> Adding the dynamic mapping (a second Kalman filter) to KFlatent changes its reconstruction little.

In [ ]:
def kfl_dyn(snr):
    S = summary(snr)
    dd = S.loc["kflatent_kf"] - S.loc["kflatent"]
    return {**{f"diff_{k}": v for k, v in dd.items()}, "all_below_0.5": bool((dd.abs() < 0.5).all())}
rows = {"main": kfl_dyn(snr), **{k: kfl_dyn(v) for k, v in variants.items() if k == "code kinematics" or k.startswith("KFlatent")}}
print(pd.DataFrame(rows).T.round(2).to_string())
print("per axis, main (dB):"); print((snr.loc["kflatent_kf"] - snr.loc["kflatent"]).round(2).to_string())

**Result: reproduced.** The dynamic map changes KFlatent's SNR by 0.18 dB for position, 0.21 for velocity and 0.09 for acceleration, all below 0.5 dB. With the other two EM seeds the changes are 0.18 to 0.19, 0.17 to 0.21 and 0.10 to 0.11 dB, and with the code's kinematics 0.29, 0.45 and 0.22 dB.

## fig2-refh-static-vel-acc

> Under the static mapping alone, velocity and acceleration are reconstructed better from the rEFH than from any other filter.

In [ ]:
OTHERS = ["linear", "kfobs", "kflatent", "kflatent_kf", "wf", "ukf"]
def static_refh(snr):
    S = summary(snr)
    out = {}
    for v in ("vel", "acc"):
        best = S.loc[OTHERS, v].idxmax()
        out[f"{v}_refh"] = S.loc["refh", v]; out[f"{v}_best_other"] = f"{best} {S.loc[best, v]:.2f}"
        out[f"{v}_margin"] = S.loc["refh", v] - S.loc[best, v]
    out["holds"] = bool(out["vel_margin"] > 0 and out["acc_margin"] > 0)
    return out
rows = {"main": static_refh(snr), **{k: static_refh(v) for k, v in variants.items()}}
print(pd.DataFrame(rows).T.to_string())

**Result: reproduced.** Among the decoders other than the rEFH (+KF), the static rEFH has the highest velocity SNR (3.87 dB against the UKF's 3.82) and acceleration SNR (1.90 against the WF's 1.34). The velocity margin is small: 0.05 dB here, 0.08 and 0.17 dB with the other seeds, and 0.09 dB over KFlatent (+KF) with the code's kinematics. The claim holds in all six runs (main, two rEFH seeds, two EM seeds, code kinematics).

## fig2-position-dynamic-gain

> Position is reconstructed much more accurately from the rEFH with the dynamic than with the static mapping, by about 3 dB of SNR.

In [ ]:
def pos_gain(snr):
    S = summary(snr)
    g = S.loc["refh_kf", "pos"] - S.loc["refh", "pos"]
    return dict(static=S.loc["refh", "pos"], dynamic=S.loc["refh_kf", "pos"], gain=g,
                gain_x=snr.loc["refh_kf", "x"] - snr.loc["refh", "x"], gain_y=snr.loc["refh_kf", "y"] - snr.loc["refh", "y"],
                within_3_pm_0_5=abs(g - 3) <= 0.5)
rows = {"main": pos_gain(snr), **{k: pos_gain(v) for k, v in variants.items()}}
print(pd.DataFrame(rows).T.round(2).to_string())

**Result: not reproduced.** The dynamic mapping raises the rEFH's position SNR from 1.46 to 5.91 dB, a gain of 4.45 dB (4.32 for x, 4.58 for y), outside 3 +- 0.5 dB. The gain is larger than the paper's, from 4.22 to 4.52 dB across the other seeds and the code's kinematics. The static map's position R2 is 0.39 and 0.51 on the training bins and 0.19 and 0.37 on the test bins (printed after the fits), so the second KF is fitted on outputs less noisy than those it filters at test.

## fig2-sagittal-better

> For each decoder and each derivative order, the sagittal (y) component is decoded better than the lateral (x) component.

In [ ]:
def sagittal(snr):
    diff = pd.DataFrame({"pos": snr["y"] - snr["x"], "vel": snr["vy"] - snr["vx"], "acc": snr["ay"] - snr["ax"]})
    return diff, int((diff > 0).values.sum())
diff, n = sagittal(snr)
print("SNR y minus x (dB), main:"); print(diff.round(2).to_string())
print(f"pairs with y > x: {n} of 24")
for k, v in variants.items():
    dv, nv = sagittal(v)
    print(f"{k}: {nv} of 24; pairs with x >= y: {[(i, c) for i in dv.index for c in dv.columns if dv.loc[i, c] <= 0]}")

**Result: partially reproduced.** The sagittal component has the higher SNR in 23 of the 24 decoder and variable pairs. The exception is the static rEFH's velocity, where y minus x is -0.06 dB. With rEFH seed 2 all 24 pairs hold; with rEFH seed 1, the two other EM seeds and the code's kinematics 23 hold, with the same exception. Linear regression's velocity margin is 0.01 dB.

## fig2-linear-worst

> Static linear regression reconstructs much worse than the other decoders.

In [ ]:
def linear_worst(snr):
    m = snr.drop(index="linear") - snr.loc["linear"]
    return m, bool((m > 0).values.all())
m, holds = linear_worst(snr)
print("SNR of each decoder minus linear regression (dB), main:"); print(m.round(2).to_string())
print("every decoder above linear regression for every variable:", holds, "; smallest margin", round(m.values.min(), 2))
for k, v in variants.items():
    mv, hv = linear_worst(v)
    print(f"{k}: holds {hv}, smallest margin {mv.values.min():.2f} dB ({mv.stack().idxmin()})")

**Result: reproduced.** Every decoder has a higher SNR than linear regression for each of the six variables. The smallest margin is 0.45 dB (static rEFH, x position), and the claim holds in every variant (smallest margin 0.20 dB, with the code's kinematics).

## fig2-gross-failures

> Gross failures (negative coefficients of determination) occur only for the UKF and KFlatent (+KF), in a few sessions.

In [ ]:
def gross(r2):
    neg = [(i, c, round(r2.loc[i, c], 3)) for i in r2.index for c in r2.columns if r2.loc[i, c] < 0]
    return neg, all(i in ("ukf", "kflatent_kf") for i, _, _ in neg)
neg, holds = gross(r2)
print("negative R2, main:", neg, "; only UKF or KFlatent (+KF):", holds)
print("lowest R2 per decoder, main:"); print(r2.min(axis=1).round(3).to_string())
neg_c, holds_c = gross(r2_c)
print("negative R2, code kinematics:", neg_c, holds_c)

**Result: reproduced, with no failures to count.** No decoder has a negative R2 at 64 ms in this session; the lowest is linear regression's 0.055. The criterion, that every negative R2 belongs to the UKF or KFlatent (+KF), therefore holds trivially. Whether failures occur in a few sessions cannot be tested on one session.

## fig4-vel-acc-every-session

> For velocity and acceleration, the rEFH (+KF) is better than KFobs and the UKF in every one of the 196 session, variable and axis combinations, and significantly so across sessions.

In [ ]:
def every(snr):
    cols = ["vx", "vy", "ax", "ay"]
    m = pd.DataFrame({r: snr.loc["refh_kf", cols] - snr.loc[r, cols] for r in ("kfobs", "ukf")}).T
    return m, int((m > 0).values.sum())
m, n = every(snr)
print("SNR rEFH (+KF) minus rival (dB), main:"); print(m.round(2).to_string())
print(f"comparisons favouring the rEFH (+KF): {n} of 8")
for k, v in variants.items():
    mv, nv = every(v)
    print(f"{k}: {nv} of 8, smallest margin {mv.values.min():.2f} dB")

**Result: reproduced for this session.** The rEFH (+KF) beats KFobs and the UKF in all 8 velocity and acceleration comparisons, by 1.04 to 1.81 dB, and in every variant (smallest margin 1.04 dB). The bootstrap across the paper's 49 sessions cannot be run on one session.

## fig4-best-variable

> The best-decoded kinematic variable is velocity for the rEFH (+KF), but position for KFobs and the UKF.

In [ ]:
def best_var(snr):
    S = summary(snr)
    b = S.loc[["refh_kf", "kfobs", "ukf"]].idxmax(axis=1)
    return S.loc[["refh_kf", "kfobs", "ukf"]], b, bool(b["refh_kf"] == "vel" and b["kfobs"] == "pos" and b["ukf"] == "pos")
Sb, b, holds = best_var(snr)
print(Sb.round(2).to_string()); print("best variable:", b.to_dict(), "; holds:", holds)
for k, v in variants.items():
    _, bv, hv = best_var(v)
    print(f"{k}: {bv.to_dict()} holds {hv}")

**Result: not reproduced.** Position is the best-decoded variable for all three decoders. The rEFH (+KF) has 5.91 dB for position against 5.06 for velocity; KFobs and the UKF have position highest, as the paper says. The ordering is the same with the other seeds and the code's kinematics.

## Single-session parts of claims blocked for the other sessions

The claims below need a bootstrap or a regression over the paper's 49 sessions, which the dandiset
does not hold, so they stay blocked. Their point comparisons can be computed on this one session
and are printed here; they do not decide an outcome.

### fig2-refh-kf-best

> Averaged over the 49 sessions, no decoder outperforms the rEFH (+KF) for any kinematic variable,
> and for velocity and acceleration it is clearly superior to all other decoders.

In [ ]:
def refh_best(snr):
    S = summary(snr)
    m = S.loc["refh_kf"] - S.drop(index="refh_kf").max()
    return {**{f"{v}_refh_kf": S.loc["refh_kf", v] for v in S.columns},
            **{f"{v}_margin_over_next": m[v] for v in S.columns},
            **{f"{v}_next": S.drop(index="refh_kf")[v].idxmax() for v in S.columns}}
rows = {"main": refh_best(snr), **{k: refh_best(v) for k, v in variants.items()}}
print(pd.DataFrame(rows).T.to_string())

### intro-gain-over-nearest

> The rEFH (+KF) improves SNR over its nearest competitor on average by 0.14 dB for position,
> 1.60 dB for velocity and 1.2 dB for acceleration; the position gain is not significant at p = 0.01.

In [ ]:
def gains(snr):
    S = summary(snr)
    a = S.loc["refh_kf"] - S.loc[OTHERS].max()
    b = S.loc["refh_kf"] - S.loc["ukf"]
    return {**{f"a_{v}": a[v] for v in S.columns}, **{f"b_{v}": b[v] for v in S.columns},
            **{f"a_rival_{v}": S.loc[OTHERS, v].idxmax() for v in S.columns}}
rows = {"main": gains(snr), **{k: gains(v) for k, v in variants.items()}}
print("expected 0.14 +- 0.014, 1.6 +- 0.16, 1.2 +- 0.12 dB (pos, vel, acc)")
print(pd.DataFrame(rows).T.to_string())

### fig4-position

> The rEFH (+KF) decodes position better than KFobs, but not distinguishably from the UKF.

In [ ]:
rows = {k: dict(minus_kfobs=summary(v).loc["refh_kf", "pos"] - summary(v).loc["kfobs", "pos"],
               minus_ukf=summary(v).loc["refh_kf", "pos"] - summary(v).loc["ukf", "pos"])
        for k, v in {"main": snr, **variants}.items()}
print("position SNR, rEFH (+KF) minus rival (dB):"); print(pd.DataFrame(rows).T.round(2).to_string())

### static-refh-position-worse

> Under the static mapping alone the rEFH reconstructs position significantly worse than the UKF,
> while the rEFH (+KF) is not distinguishable from it.

In [ ]:
rows = {k: dict(static_refh=summary(v).loc["refh", "pos"], ukf=summary(v).loc["ukf", "pos"],
               static_refh_minus_ukf=summary(v).loc["refh", "pos"] - summary(v).loc["ukf", "pos"])
        for k, v in {"main": snr, **variants}.items()}
print("position SNR (dB):"); print(pd.DataFrame(rows).T.round(2).to_string())

**Result: blocked, single-session values only.** These claims need the bootstrap over the paper's
49 sessions. On this session:
- fig2-refh-kf-best: the rEFH (+KF) has the highest SNR for position, velocity and acceleration
  (5.91, 5.06 and 2.36 dB), ahead of the next decoder by 0.28 dB (UKF), 1.19 and 0.45 dB (static
  rEFH); it is highest in every variant.
- intro-gain-over-nearest: the gains over the nearest rival are 0.28, 1.24 and 1.02 dB under
  reading a (UKF, UKF, WF) and 0.28, 1.24 and 1.25 dB under reading b (UKF). The position and
  velocity gains are outside 0.14 +- 0.014 and 1.6 +- 0.16 dB under both readings and with every
  seed (position 0.11 to 0.28 dB, velocity 1.24 to 1.31 dB). With the code's kinematics the
  velocity gain is inside (1.45 and 1.55 dB) but the position gain is 0.51 dB.
- fig4-position: the rEFH (+KF) position SNR exceeds KFobs's by 0.90 dB and the UKF's by 0.28 dB
  (0.73 to 1.02 and 0.11 to 0.51 dB across the variants).
- static-refh-position-worse: the static rEFH's position SNR is 4.17 dB below the UKF's (1.46
  against 5.63), and 4.01 to 4.12 dB below in the variants.

## kflatent-modest

> KFlatent is only modestly better than the standard Kalman filter; nonlinear tuning (the UKF) improves on the standard filter more.

In [ ]:
def kfl_modest(snr):
    S = summary(snr)
    reads = {"a velocity": ["vel"], "b velocity and acceleration": ["vel", "acc"], "c all three": ["pos", "vel", "acc"]}
    out = {}
    for name, cols in reads.items():
        kfl = max(S.loc["kflatent", cols].mean(), S.loc["kflatent_kf", cols].mean())
        kfo, ukf = S.loc["kfobs", cols].mean(), S.loc["ukf", cols].mean()
        out[name] = dict(kflatent_better_minus_kfobs=kfl - kfo, ukf_minus_kfobs=ukf - kfo,
                         holds=bool(kfl > kfo and (ukf - kfo) > (kfl - kfo)))
    return pd.DataFrame(out).T
print("main:"); print(kfl_modest(snr).to_string())
for k, v in variants.items():
    if k == "code kinematics" or k.startswith("KFlatent"):
        print(f"{k}:"); print(kfl_modest(v).to_string())
print("per variable, main (dB):"); print(summary(snr).loc[["kfobs", "kflatent", "kflatent_kf", "ukf"]].round(2).to_string())

**Result: reproduced.** On velocity (reading a), the better KFlatent decoder exceeds KFobs by 0.10 dB while the UKF exceeds it by 0.51 dB; on velocity and acceleration (reading b) by 0.08 and 0.30 dB. Over all three variables (reading c) KFlatent is 0.20 dB below KFobs. With the two other EM seeds KFlatent's lead over KFobs is 0.08 and 0.08 dB under reading a and 0.07 and 0.08 dB under reading b, so readings a and b hold for all three EM runs. With the code's kinematics readings a and b fail (KFlatent ahead of KFobs by 0.49 and 0.39 dB, more than the UKF's 0.38 and 0.22) and reading c holds (0.07 against 0.32 dB). Under either kinematics the claim holds under one reading.

## fig8-velocity-variants

> Under the static mapping, about 0.7 dB of the rEFH's velocity advantage over the UKF is lost
> when the updated visible units are left out of decoding, and 0.4 dB when standard-normal
> visible units replace Poisson ones; Kalman filtering does not remove these differences.

The variant without visible units decodes the canonical rEFH from its hidden means alone. The
standard-normal rEFH is trained on standardized counts (the Results, "Variant decoders"); the
code's transform 2 sqrt(R + 3/8) is the variant the criterion names.

In [ ]:
t = time.time()
sp8 = M.prepare(d)
canon = M.dec_refh(sp8, seed=1, variants=True)          # seed 1 is the main rEFH run (run_all uses seed + 1)
norm_std = M.dec_refh(sp8, seed=1, units="normal", variants="standardize")
norm_ans = M.dec_refh(sp8, seed=1, units="normal", variants="anscombe")
print(f"variants fitted in {time.time() - t:.0f} s")
v8 = pd.DataFrame({
    "canonical": sp8.score(canon["refh"]), "canonical +KF": sp8.score(canon["refh_kf"]),
    "hidden only": sp8.score(canon["refh_hid"]), "hidden only +KF": sp8.score(canon["refh_hid_kf"]),
    "std normal (standardized)": sp8.score(norm_std["refh"]), "std normal (standardized) +KF": sp8.score(norm_std["refh_kf"]),
    "std normal (code transform)": sp8.score(norm_ans["refh"]), "std normal (code transform) +KF": sp8.score(norm_ans["refh_kf"]),
    "ukf": r2.loc["ukf"].values}, index=M.VARS).T
S8 = summary(M.snr(v8))
print(S8.round(2).to_string())
vel = S8["vel"]
loss_hid = vel["canonical"] - vel["hidden only"]
loss_std = vel["canonical"] - vel["std normal (standardized)"]
loss_ans = vel["canonical"] - vel["std normal (code transform)"]
print(f"\nstatic velocity loss, hidden only: {loss_hid:.2f} dB (text 0.7 +- 0.07; figure about 0.3)")
print(f"static velocity loss, standard normal (standardized): {loss_std:.2f} dB (text 0.4 +- 0.04; figure about 0.4)")
print(f"static velocity loss, standard normal (code transform): {loss_ans:.2f} dB")
print("advantage of canonical static over UKF, velocity:", round(vel["canonical"] - vel["ukf"], 2))
for nm in ["hidden only +KF", "std normal (standardized) +KF", "std normal (code transform) +KF"]:
    print(f"dynamic velocity, canonical +KF minus {nm}: {vel['canonical +KF'] - vel[nm]:.2f} dB")

**Result: not reproduced.** Leaving the updated visible units out costs 0.11 dB of static velocity SNR (the text says 0.7 dB, figure 8 about 0.3). The standard-normal rEFH trained on standardized counts gains 0.08 dB (a loss of -0.08 dB against the text's 0.4). Trained on the code's transform, 2 sqrt(R + 3/8), it loses 2.11 dB. Under the dynamic mapping the canonical rEFH (+KF) still exceeds the variants, by 0.15, 0.15 and 1.68 dB. The static rEFH's own velocity advantage over the UKF is only 0.05 dB here.

## fig8-gaussian-still-beats-ukf

> The rEFH keeps an advantage over the UKF when its visible units are standard normal instead of Poisson, so the noise model accounts for only part of its advantage.

In [ ]:
for nm in ["std normal (standardized) +KF", "std normal (code transform) +KF"]:
    dv = S8.loc[nm] - S8.loc["ukf"]
    print(f"{nm} minus UKF (dB): vel {dv['vel']:.2f}, acc {dv['acc']:.2f}, pos {dv['pos']:.2f}; vel and acc above UKF: {bool(dv['vel'] > 0 and dv['acc'] > 0)}")

**Result: reproduced, resting on the standardized-counts reading.** Trained on standardized counts, the standard-normal rEFH (+KF) exceeds the UKF by 1.09 dB for velocity and 1.17 dB for acceleration. Trained on the code's transform it is 0.43 dB below the UKF for velocity (0.12 dB above for acceleration), and the claim fails.

## fig8-position-variants (single-session part)

> For position the rEFH variants change the static but not the dynamic mapping; with the Kalman
> filter all three variants decode position about equally.

The dynamic part needs the bootstrap over sessions; the static part and the dynamic differences
are printed.

In [ ]:
pos = S8["pos"]
print(pos.round(2).to_string())
print(f"static: canonical minus hidden only {pos['canonical'] - pos['hidden only']:.2f} dB, minus standard normal (standardized) "
      f"{pos['canonical'] - pos['std normal (standardized)']:.2f} dB, minus standard normal (code transform) {pos['canonical'] - pos['std normal (code transform)']:.2f} dB")
for nm in ["hidden only +KF", "std normal (standardized) +KF", "std normal (code transform) +KF"]:
    print(f"dynamic: canonical +KF minus {nm}: {pos['canonical +KF'] - pos[nm]:.2f} dB")

**Result: blocked, single-session values only.** Under the static mapping the canonical rEFH has
a higher position SNR (1.46 dB) than both variants (hidden units only 1.43, standard normal on
standardized counts 1.38, on the code's transform 0.99), so the static part holds. Under the
dynamic mapping the canonical rEFH (+KF) is 0.11 and 0.16 dB above the two variants (1.17 dB above
the code-transform variant); whether that is below p < 0.01 needs the bootstrap over sessions.

## Summary

| Claim | Outcome |
|---|---|
| units-monkey-i | not reproduced (111 units; no hash units in the file) |
| fig2-velocity-boost | not reproduced, resting on the Methods kinematics (reproduced with the code's) |
| fig2-kflatent-dynamic-little-effect | reproduced |
| fig2-refh-static-vel-acc | reproduced |
| fig2-position-dynamic-gain | not reproduced (gain 4.45 dB, not about 3) |
| fig2-sagittal-better | partially reproduced (23 of 24) |
| fig2-linear-worst | reproduced |
| fig2-gross-failures | reproduced (no negative R2 at 64 ms; other configurations have some, see `binwidth_training_neurons.ipynb`) |
| fig4-vel-acc-every-session | reproduced for this session (bootstrap not run) |
| fig4-best-variable | not reproduced (the rEFH (+KF) decodes position best) |
| kflatent-modest | reproduced |
| fig8-velocity-variants | not reproduced |
| fig8-gaussian-still-beats-ukf | reproduced, resting on standardized counts |
| fig5-advantage-with-binwidth, fig5-derivative-order, advantage-grows-with-derivative, fig7-affine-position, fig7-kinematic-model-saturates | see `binwidth_training_neurons.ipynb` |
| fig2-refh-kf-best, intro-gain-over-nearest, fig4-advantage-grows, fig4-position, fig5-vel-acc-all-bins, fig5-position-bins, fig6-training-time, fig7-vel-acc-neurons, fig7-position-neurons, fig8-position-variants, fig9-taps, static-refh-position-worse | blocked: they need the paper's other sessions, which the dandiset does not hold (single-session values printed above and in `binwidth_training_neurons.ipynb`) |
| fig4-task-similar | blocked: monkey C's sessions have no archive identifier, and the dandiset's one monkey I session is probably not one of the paper's |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** One session in place of 49, so every mean over
sessions is this session's value and no bootstrap is run. The session is dated 2017-02-02, which
is not in the paper's session list. Sorted units only (111 above 0.5 Hz), without the hash units
the paper kept. The test part is the last 329 s of the training file; the test file's kinematics
are withheld. Six bins that overlap the file's three 0.1 s gaps are left out of fitting and
scoring. All decoders are re-implemented in Python from the Methods and the paper's code; the
UKF re-implements Li's UKF2, whose fitting code is not in the paper's repository. The rEFH is
trained in single precision. EM for KFlatent ran the full 100 epochs without meeting the
convergence criterion of 1/100 of a percent, as the Methods allow. The rEFH and the
factor-analysis initialization of EM are each run with three seeds. The Wiener filter takes the
history of the first test bins from the last training bins.

**What was found about the dataset.** The dandiset holds one of the paper's three monkeys and one
session, which the paper's session list does not include. It has no hash units. Three 0.1 s gaps
split the training file into four segments, with NaN kinematics and no spikes, and nothing says
what was cut. The test file's 190 s have no kinematics. The units and the kinematics share one
clock: velocity is best decoded from spikes 64 ms earlier.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")